# Demo: `ts_compress_to` with Sequence Numbers

This notebook demonstrates the compression table operator on a permanent
Teradata table with a BIGINT sequence column. It generates varied time series,
loads them to Teradata, installs the C table operator, runs compression,
measures runtime, validates the selected rows, and overlays compressed points
on the original signal.

## 1. Setup

Credentials are loaded from `.env` with `load_dotenv`. Required variables are
`TD_HOST`, `TD_USERNAME`, and `TD_PASSWORD`; optional variables are
`TD_DATABASE` and `TD_LOGMECH`.

In [ ]:
from pathlib import Path
import os
import time

import td_ts_compressor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import teradatasql
from dotenv import load_dotenv
from IPython.display import Image, display

from td_ts_compressor import (
    __version__ as TD_TS_COMPRESSOR_VERSION,
    CompressorSpec,
    TeradataTimeSeriesCompressor,
    analytical_points,
    visual_fidelity,
)

pio.renderers.default = "notebook"

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pyproject.toml").exists() else NOTEBOOK_DIR.parent
ENV_PATH = REPO_ROOT / ".env"
FIGURE_DIR = REPO_ROOT / "notebooks" / "assets"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TD_TS_COMPRESSOR_VERSION = "0.1.10"
if TD_TS_COMPRESSOR_VERSION != EXPECTED_TD_TS_COMPRESSOR_VERSION:
    raise RuntimeError(
        "Notebook expects td-ts-compressor "
        f"{EXPECTED_TD_TS_COMPRESSOR_VERSION}, but imported {TD_TS_COMPRESSOR_VERSION}. "
        "Reinstall with: python -m pip install -e . and restart the kernel."
    )

load_dotenv(ENV_PATH)

required = ["TD_HOST", "TD_USERNAME", "TD_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

OPERATOR_DATABASE = os.getenv("TD_OPERATOR_DATABASE") or os.getenv("TD_DATABASE")

print(f"Repository root: {REPO_ROOT}")
print(f"td_ts_compressor version: {TD_TS_COMPRESSOR_VERSION}")
print(f"td_ts_compressor path: {Path(td_ts_compressor.__file__).resolve()}")
print("Operator database:", OPERATOR_DATABASE or "current session database")




## 2. Connect to Teradata

In [ ]:
connect_args = {
    "host": os.environ["TD_HOST"],
    "user": os.environ["TD_USERNAME"],
    "password": os.environ["TD_PASSWORD"],
}
if os.getenv("TD_LOGMECH"):
    connect_args["logmech"] = os.environ["TD_LOGMECH"]

con = teradatasql.connect(**connect_args)
cur = con.cursor()
if os.getenv("TD_DATABASE"):
    cur.execute(f"DATABASE {os.environ['TD_DATABASE']}")
cur.execute("SELECT DATABASE")
CURRENT_DATABASE = cur.fetchone()[0]
print("Current database:", CURRENT_DATABASE)


## 3. Generate and Visualize Input Data

The dataset contains 40 series with 1,000 rows each. The patterns include
trends, periodic signals, peaks, noisy walks, step changes, and mixed signals.

In [ ]:
rng = np.random.default_rng(42)
n_series = 40
n_rows_per_series = 1_000
seq = np.arange(1, n_rows_per_series + 1, dtype=np.int64)
t = np.linspace(0, 1, n_rows_per_series)

def make_series(series_idx: int) -> tuple[str, np.ndarray]:
    family = series_idx % 6
    noise = rng.normal(0, 0.001 + 0.0002 * (series_idx % 5), n_rows_per_series)
    base = 2.0 * (series_idx // 6)
    if family == 0:
        label = "trend"
        y = base + 8.0 * t + noise
    elif family == 1:
        label = "periodic"
        y = base + 3.0 * np.sin(2 * np.pi * (3 + series_idx % 4) * t) + noise
    elif family == 2:
        label = "peaks"
        y = base + noise
        for center in rng.choice(seq, size=8, replace=False):
            y += rng.uniform(4, 10) * np.exp(-0.5 * ((seq - center) / rng.uniform(5, 25)) ** 2)
    elif family == 3:
        label = "noisy_walk"
        y = base + np.cumsum(rng.normal(0, 0.004, n_rows_per_series)) + noise
    elif family == 4:
        label = "step_change"
        y = base + np.where(seq < 350, 0.0, np.where(seq < 700, 4.0, -1.5)) + noise
    else:
        label = "mixed"
        y = base + 5.0 * t + 2.0 * np.sin(2 * np.pi * 5 * t) + rng.normal(0, 0.004, n_rows_per_series)
    return label, y.astype(float)

frames = []
for series_idx in range(n_series):
    pattern, y = make_series(series_idx)
    frames.append(pd.DataFrame({
        "series_id_1": pattern,
        "series_id_2": series_idx + 1,
        "seq_no": seq,
        "y": y,
    }))
input_df = pd.concat(frames, ignore_index=True)
print(input_df.shape)
input_df.head()


In [ ]:
sample_ids = [1, 2, 3, 4, 5, 6, 13, 24]
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
for ax, series_id_2 in zip(axes.ravel(), sample_ids):
    sample = input_df[input_df["series_id_2"] == series_id_2]
    ax.plot(sample["seq_no"], sample["y"], linewidth=1.0)
    ax.set_title(f"{sample['series_id_1'].iloc[0]} / series {series_id_2}")
    ax.grid(alpha=0.25)
fig.suptitle("Original generated time series", y=1.02)
fig.tight_layout()
figure_path = FIGURE_DIR / "03_original_series.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))


## 4. Load a Permanent Teradata Table

In [ ]:
def execute_ignore_error(sql: str) -> None:
    try:
        cur.execute(sql)
    except Exception as exc:
        if "3807" not in str(exc):
            print(f"Ignoring cleanup error: {exc}")

execute_ignore_error("DROP TABLE input_timeseries_compress")
cur.execute("""
CREATE TABLE input_timeseries_compress
(
    series_id_1 VARCHAR(128) CHARACTER SET LATIN,
    series_id_2 INTEGER,
    seq_no      BIGINT,
    y           FLOAT
)
PRIMARY INDEX (series_id_1, series_id_2)
""")
cur.executemany(
    "INSERT INTO input_timeseries_compress VALUES (?, ?, ?, ?)",
    list(input_df.itertuples(index=False, name=None)),
)
con.commit()
cur.execute("SELECT COUNT(*) FROM input_timeseries_compress")
print("Rows inserted:", cur.fetchone()[0])


## 5. Configure Package SQL Builder

The table operator is installed once by `00_install_ts_compress_to_operator.ipynb`. This demo relies on the package to generate operator SQL and uses the installed mapping, preserving the input columns plus `compression_importance` and `compression_distance`.


In [ ]:
compressor = TeradataTimeSeriesCompressor(
    CompressorSpec(
        id_columns=["series_id_1", "series_id_2"],
        time_column="seq_no",
        value_column="y",
        time_type="numeric",
        time_unit="second",
    ),
    operator_database=OPERATOR_DATABASE,
)

print("Using mapping:", f"{OPERATOR_DATABASE + '.' if OPERATOR_DATABASE else ''}ts_compress_seq_to_map")



## 6. Run Compression and Measure Duration

This notebook demonstrates the two main ways to use the same compression operator on the same input table:

- **Visual-fidelity compression**: `distance_max('1.0')`, `n_max('-1')`, `tol_x('auto')`, and `tol_y('auto')`. The automatic tolerance is calibrated on a small 100x100 plot space so sub-pixel noise is ignored and only structural deviations are retained.
- **Analytical point extraction**: `n_max('auto')`, `distance_max('-1')`, and `auto_log_drop('1.5')`. This extracts the most important points by following the residual-distance curve rather than targeting a plotting resolution.

Both modes use the actual `seq_no` values as x, so irregular sequence gaps would affect the geometry. The dataset includes a small, sub-pixel amount of measurement noise for smooth families so the operator is tested on realistic but still readable curves without chasing noise.

In [ ]:
mode_specs = {
    "visual_fidelity": visual_fidelity(width_px=100, height_px=100),
    "analytical_points": analytical_points(auto_log_drop=1.5).with_overrides(
        plot_width_px=100,
        plot_height_px=100,
    ),
}

source_columns = ["series_id_1", "series_id_2", "seq_no", "y"]

def build_operator_sql(mode_name: str, parameters) -> str:
    base_sql = compressor.compress(
        table_name="input_timeseries_compress",
        parameters=parameters,
        columns=source_columns,
        as_dataframe=False,
    )
    return f"""
SELECT
    CAST('{mode_name}' AS VARCHAR(64)) AS compression_mode,
    t.*
FROM (
{base_sql}
) AS t
"""

compressed_by_mode = {}
for mode_name, parameters in mode_specs.items():
    operator_sql = build_operator_sql(mode_name, parameters)
    start = time.perf_counter()
    cur.execute(operator_sql + " ORDER BY series_id_1, series_id_2, seq_no")
    rows = cur.fetchall()
    elapsed = time.perf_counter() - start
    compressed_by_mode[mode_name] = pd.DataFrame(
        rows,
        columns=[
            "compression_mode",
            "series_id_1",
            "series_id_2",
            "seq_no",
            "y",
            "compression_importance",
            "compression_distance",
        ],
    )
    print(
        f"{mode_name}: {len(compressed_by_mode[mode_name]):,} / "
        f"{len(input_df):,} rows; duration: {elapsed:.3f} seconds"
    )

visual_fidelity_df = compressed_by_mode["visual_fidelity"]
analytical_points_df = compressed_by_mode["analytical_points"]

visual_summary = visual_fidelity_df.groupby(["series_id_1", "series_id_2"]).size().rename("visual_points").reset_index()
analytical_summary = analytical_points_df.groupby(["series_id_1", "series_id_2"]).size().rename("analytical_points").reset_index()

print("Visual-fidelity compression sample")
display(visual_fidelity_df.head(5))
display(visual_summary.head(8))

print("Analytical point extraction sample")
display(analytical_points_df.head(5))
display(analytical_summary.head(8))



## 7. Feature Engineering with `compressor.features_sql()`

The operator only returns selected rows. The package method `compressor.features_sql()` wraps that operator output in downstream SQL that computes reusable feature vectors: `compressed_points`, `compression_ratio`, `reduction_pct`, distance summaries, and optional top-k distances by `compression_importance`. These features can feed clustering or monitoring without changing the C table operator.


In [ ]:
FEATURE_TOP_K = 3

feature_sql = compressor.features_by_mode_sql(
    table_name="input_timeseries_compress",
    modes=mode_specs,
    columns=source_columns,
    top_k_distances=FEATURE_TOP_K,
    mode_column="mode_raw",
    order_by=["series_id_2", "mode_raw"],
)

start = time.perf_counter()
cur.execute(feature_sql)
feature_rows = cur.fetchall()
feature_elapsed = time.perf_counter() - start

feature_columns = [
    "mode_raw",
    "series_id_1",
    "series_id_2",
    "total_points",
    "remaining_points",
    "compressed_points",
    "retained_ratio",
    "retained_pct",
    "compression_ratio",
    "reduction_pct",
    "max_selected_distance",
    "avg_selected_distance",
    "std_selected_distance",
    *[f"distance_rank_{rank}" for rank in range(1, FEATURE_TOP_K + 1)],
]
feature_sql_df = pd.DataFrame(feature_rows, columns=feature_columns)
feature_sql_df["mode_key"] = feature_sql_df["mode_raw"].astype(str).str.strip().str.lower()
feature_sql_df.loc[feature_sql_df["mode_key"].str.startswith("analytical", na=False), "mode_key"] = "analytical_points"
feature_sql_df.loc[feature_sql_df["mode_key"].str.startswith("visual", na=False), "mode_key"] = "visual_fidelity"
feature_sql_df["mode"] = feature_sql_df["mode_key"].map({
    "visual_fidelity": "visual fidelity",
    "analytical_points": "analytical points",
}).fillna(feature_sql_df["mode_key"])
feature_sql_df["series_id_1"] = feature_sql_df["series_id_1"].astype(str).str.strip()
feature_sql_df["series_label"] = feature_sql_df["series_id_2"].astype(str) + " - " + feature_sql_df["series_id_1"]

numeric_columns = [
    "total_points",
    "remaining_points",
    "compressed_points",
    "retained_ratio",
    "retained_pct",
    "compression_ratio",
    "reduction_pct",
    "max_selected_distance",
    "avg_selected_distance",
    "std_selected_distance",
    *[f"distance_rank_{rank}" for rank in range(1, FEATURE_TOP_K + 1)],
]
for column in numeric_columns:
    feature_sql_df[column] = pd.to_numeric(feature_sql_df[column], errors="coerce")

print(f"Package feature SQL returned {len(feature_sql_df)} rows in {feature_elapsed:.3f} seconds")
print("Feature rows by mode:")
display(feature_sql_df.groupby("mode", dropna=False).size().rename("rows").reset_index())
display(feature_sql_df.head(12))

plot_features_df = feature_sql_df[feature_sql_df["mode_key"] == "analytical_points"].copy()
if plot_features_df.empty:
    available_modes = ", ".join(sorted(feature_sql_df["mode_key"].dropna().unique()))
    raise RuntimeError(f"No analytical_points rows to plot. Available modes: {available_modes}")

family_palette = {
    "trend": "#2a9d8f",
    "step_change": "#ef4444",
    "periodic": "#3b82f6",
    "peaks": "#c56a24",
    "noisy_walk": "#7b8492",
    "mixed": "#8b5cf6",
}
family_labels = {
    "trend": "trend",
    "step_change": "step",
    "periodic": "periodic",
    "peaks": "peaks",
    "noisy_walk": "noisy",
    "mixed": "mixed",
}
family_order = ["trend", "step_change", "periodic", "peaks", "noisy_walk", "mixed"]
label_offsets = {
    "Compression feature space": {
        "trend": (-36, -2),
        "step_change": (18, -4),
        "periodic": (18, -16),
        "peaks": (12, 18),
        "noisy_walk": (14, -4),
        "mixed": (10, 18),
    },
    "Output complexity feature space": {
        "trend": (8, 2),
        "step_change": (8, 14),
        "periodic": (8, -8),
        "peaks": (10, 8),
        "noisy_walk": (10, -8),
        "mixed": (10, 10),
    },
}

plt.style.use("default")
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2), dpi=150)
fig.patch.set_facecolor("white")

plot_specs = [
    (axes[0], "reduction_pct", "max_selected_distance", "Compression feature space", "reduction_pct: rows removed (%)", "max_selected_distance"),
    (axes[1], "compressed_points", "std_selected_distance", "Output complexity feature space", "compressed_points", "std_selected_distance"),
]

for ax, x_col, y_col, title, x_label, y_label in plot_specs:
    ax.set_facecolor("white")
    ax.grid(True, color="#d9dee7", linewidth=1.0, alpha=0.75)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#111827")
    ax.spines["bottom"].set_color("#111827")
    ax.spines["left"].set_linewidth(1.15)
    ax.spines["bottom"].set_linewidth(1.15)
    ax.tick_params(colors="#6b7280", labelsize=10)
    ax.set_title(title, loc="left", fontsize=17, fontweight="bold", color="#111827", pad=8)
    ax.set_xlabel(x_label, fontsize=12, color="#111827")
    ax.set_ylabel(y_label, fontsize=12, color="#111827")

    for family in family_order:
        family_df = plot_features_df[plot_features_df["series_id_1"] == family]
        if family_df.empty:
            continue
        ax.scatter(
            family_df[x_col],
            family_df[y_col],
            s=54,
            color=family_palette[family],
            alpha=0.86,
            edgecolor="white",
            linewidth=0.55,
            label=family_labels[family],
        )
        if title == "Compression feature space":
            label_row = family_df.sort_values([x_col, y_col]).iloc[-1]
        else:
            label_row = family_df.sort_values([x_col, y_col]).iloc[len(family_df) // 2]
        text_offset = label_offsets[title].get(family, (8, 6))
        ax.annotate(
            family_labels[family],
            xy=(label_row[x_col], label_row[y_col]),
            xytext=text_offset,
            textcoords="offset points",
            color=family_palette[family],
            fontsize=12,
            fontweight="bold",
            alpha=0.95,
            zorder=5,
            bbox=dict(boxstyle="round,pad=0.12", facecolor="white", edgecolor="none", alpha=0.58),
        )

axes[0].set_xlim(plot_features_df["reduction_pct"].min() - 4, min(100.5, plot_features_df["reduction_pct"].max() + 3))
axes[0].set_ylim(bottom=min(0, plot_features_df["max_selected_distance"].min() - 0.1))
axes[1].set_xlim(left=max(0, plot_features_df["compressed_points"].min() - 8))
axes[1].set_ylim(bottom=min(0, plot_features_df["std_selected_distance"].min() - 0.05))

handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markerfacecolor=family_palette[family],
        markeredgecolor="white",
        markeredgewidth=0.55,
        markersize=8,
        label=family_labels[family],
    )
    for family in family_order
    if family in set(plot_features_df["series_id_1"])
]
fig.legend(
    handles=handles,
    loc="lower center",
    ncol=len(handles),
    frameon=False,
    fontsize=11,
    bbox_to_anchor=(0.5, -0.02),
)
fig.text(
    0.5,
    0.055,
    "Illustrative feature vectors computed in SQL from compression output: count, compression ratio/reduction, and distance summaries.",
    ha="center",
    va="center",
    fontsize=10.5,
    color="#6b7280",
)
fig.tight_layout(rect=[0, 0.09, 1, 1])

feature_figure_path = FIGURE_DIR / "04_feature_engineering_static.png"
fig.savefig(feature_figure_path, dpi=160, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(feature_figure_path)))


## 8. tdfs4ds Process View and Deterministic Documentation

For feature-store workflows, `process_feature_engineering()` wraps one compressor mode as a persisted tdfs4ds process view and returns the two business dictionary dataframes expected by tdfs4ds. The method uses `process_database` and `process_name` for the generated view, applies an optional prefix or suffix to generated feature columns only, and can read existing tdfs4ds documentation for the source object and input columns when tdfs4ds is installed and connected. That source documentation is then used to replace generic language such as "series" with the actual business meaning of the signal.


In [ ]:
PROCESS_DATABASE = os.getenv("TD_PROCESS_DATABASE") or os.getenv("TD_DATABASE") or CURRENT_DATABASE
PROCESS_NAME = "vw_feat_ts_compress_demo"
if not PROCESS_DATABASE:
    raise RuntimeError("Set TD_PROCESS_DATABASE or TD_DATABASE, or run the Teradata connection cell first.")
print(f"Process view: {PROCESS_DATABASE}.{PROCESS_NAME}")

process_artifact = compressor.process_feature_engineering(
    process_database=PROCESS_DATABASE,
    process_name=PROCESS_NAME,
    source_database=CURRENT_DATABASE,
    source_object="input_timeseries_compress",
    parameters=analytical_points(),
    mode_name="analytical_points",
    top_k_distances=FEATURE_TOP_K,
    feature_prefix="ts_",
    series_description="synthetic signal profile",
    source_object_description=(
        "Synthetic time-series table used to demonstrate trend, step-change, "
        "periodic, peak-heavy, noisy-walk, and mixed signal families."
    ),
    source_column_descriptions={
        "series_id_1": "Signal family label for the synthetic input series.",
        "series_id_2": "Numeric identifier of one generated signal within its family.",
        "seq_no": "Ordered sample number within each generated signal.",
        "y": "Observed synthetic signal value at the sample position.",
    },
    use_tdfs4ds_dictionary=False,
)

print(process_artifact.process_sql[:900] + "\n...")
print(process_artifact.feature_names)
display(process_artifact.object_df)
display(process_artifact.columns_df.head(12))

# To persist and register the process in a connected tdfs4ds workflow:
# cur.execute(process_artifact.process_sql)
# df_features_proc = tdml.DataFrame(tdml.in_schema(PROCESS_DATABASE, PROCESS_NAME))
# upload_features(
#     df_features_proc,
#     entity_id={"series_id_1": "VARCHAR(32)", "series_id_2": "INTEGER"},
#     feature_names=process_artifact.feature_names,
#     metadata=process_artifact.metadata,
#     dataset_view_name="ds_ts_compress_demo",
# )
# upload_business_dictionary_objects(process_artifact.object_df)
# upload_business_dictionary_columns(process_artifact.columns_df)


## 9. Validate and Visualize Compression


In [ ]:
cur.execute("SELECT COUNT(*) FROM input_timeseries_compress")
input_count = cur.fetchone()[0]
for mode_name, compressed_df in compressed_by_mode.items():
    assert len(compressed_df) < input_count, f"{mode_name} should return a subset of rows"
    assert compressed_df.groupby(["series_id_1", "series_id_2"]).size().min() >= 2
print("Validation passed: both use cases return subsets with at least endpoints per series.")


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
for ax, series_id_2 in zip(axes.ravel(), sample_ids):
    original = input_df[input_df["series_id_2"] == series_id_2]
    visual_fidelity = visual_fidelity_df[visual_fidelity_df["series_id_2"] == series_id_2]
    analytical_points = analytical_points_df[analytical_points_df["series_id_2"] == series_id_2]
    ax.plot(original["seq_no"], original["y"], linewidth=0.8, alpha=0.45, label="original")
    ax.plot(
        visual_fidelity["seq_no"],
        visual_fidelity["y"],
        linewidth=1.1,
        marker="o",
        markersize=2.2,
        label=f"visual fidelity ({len(visual_fidelity)})",
    )
    ax.plot(
        analytical_points["seq_no"],
        analytical_points["y"],
        linewidth=1.3,
        marker="s",
        markersize=2.2,
        label=f"analytical points ({len(analytical_points)})",
    )
    ax.set_title(f"{original['series_id_1'].iloc[0]} / series {series_id_2}")
    ax.grid(alpha=0.25)
    ax.legend(loc="best", fontsize=8)
fig.suptitle("Visual-fidelity compression and analytical point extraction", y=1.02)
fig.tight_layout()
figure_path = FIGURE_DIR / "03_compression_use_cases.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))


## 10. Cleanup

In [ ]:
cur.close()
con.close()
print("Connection closed")
